In [1]:
# use cvTransPCA to run 3 scenarios based on r_k^s = r_0^s
import numpy as np
import pandas as pd
from tqdm import tqdm
import random
import seaborn as sns
from scipy.stats import ortho_group

from matrix_completion import * 
from sklearn.linear_model import LinearRegression
from itertools import chain
from sklearn.model_selection import KFold
from pandas.core.frame import DataFrame
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

import torch
import math
import gc
from TransMC_sensitivity import *  

In [2]:
if __name__ == "__main__":
    K = 10
    # parameters
    params = {
        'scenario': 0,                              # R and C are useful
        'p': 50,                                    # the row number of matrices
        'q': 50,                                    # the column number of matrices
        'p0': 5,                                    # the dimension of the row central subspace
        'q0': 5,                                    # the dimension of the column central subspace
        'K': K,                                     # total dataset number (one target + K-1 sources)
        'K_useful': int(K/2),                       # the number of useful sources
        'K_useless_R': int(K/6),                    # the number of sources with only useful row spaces
        'K_useless_C': int(K/6),                    # the number of sources with only useful column spaces
        'K_useless_RC': K - int(K/2) - int(K/6) - int(K/6), # the number of sources with different row and column spaces
        'ri': 3,                                    # the dimension for local Theta matrix
        'h': 0.01,                                  # similarity level
        'Sscale': 1,                                # the scale level of the signal
        'n_fold': 300,                              # sample size for each fold in each source
        'k_fold': 3,                                # the input fold number for generate_kfold_observations
        'tau_range': 3 - np.linspace(1, 2.4, num=15), # the searching grid for tau
        'lambda_range': np.logspace(-2, 0.7, num=15), # the searching grid for lambda
        'iterate': 10,                              # iteration time
        'T': 3,                                     # iteration number for kmeans
        'n_iter': 3                                 # iteration number for penalization
    }

In [3]:
# activate main function
Error_average = run_simulation_nonoracle_nocenter(**params)
np.save("Error_average.npy", Error_average)

 (tau, lambda) = (2.000000, 0.010000) start
 (tau, lambda) = (2.000000, 0.015590) start
 (tau, lambda) = (2.000000, 0.024306) start
 (tau, lambda) = (2.000000, 0.037894) start
 (tau, lambda) = (2.000000, 0.059078) start
 (tau, lambda) = (2.000000, 0.092106) start
 (tau, lambda) = (2.000000, 0.143596) start
 (tau, lambda) = (2.000000, 0.223872) start
 (tau, lambda) = (2.000000, 0.349025) start
 (tau, lambda) = (2.000000, 0.544145) start
 (tau, lambda) = (2.000000, 0.848343) start
 (tau, lambda) = (2.000000, 1.322600) start
 (tau, lambda) = (2.000000, 2.061986) start
 (tau, lambda) = (2.000000, 3.214718) start
 (tau, lambda) = (2.000000, 5.011872) start
 (tau, lambda) = (1.900000, 0.010000) start
 (tau, lambda) = (1.900000, 0.015590) start
 (tau, lambda) = (1.900000, 0.024306) start
 (tau, lambda) = (1.900000, 0.037894) start
 (tau, lambda) = (1.900000, 0.059078) start
 (tau, lambda) = (1.900000, 0.092106) start
 (tau, lambda) = (1.900000, 0.143596) start
 (tau, lambda) = (1.900000, 0.22

In [4]:
# draw the heatmap
colors = ["#2166AC", "#F7F7F7", "#B2182B"]  # blue-white-red
cmap = LinearSegmentedColormap.from_list("custom_div", colors)

# parameter range
tau_range_raw = np.linspace(1, 2.4, num=15)
lambda_range = np.logspace(-2, 0.7, num=15)

Error_plot = Error_average
tau_plot = tau_range_raw
lambda_plot_raw = lambda_range

vmin = Error_plot.min()
vmax = Error_plot.max()

if vmin < 0 < vmax:
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
else:
    norm = None
cbar_ticks = np.linspace(vmin, vmax, 6)
plt.figure(figsize=(8, 7))
heatmap = sns.heatmap(
    np.flipud(Error_plot.T),
    annot=False,
    cmap=cmap,
    norm=norm,
    vmin=None if norm is not None else vmin,
    vmax=None if norm is not None else vmax,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "ticks": cbar_ticks,
        "format": "%.2f"
    }
)

cbar = heatmap.collections[0].colorbar
cbar.ax.tick_params(labelsize=14)

# axis tick number
n_ticks = 5

# x-axis: tau
x_tick_indices = np.linspace(0, len(tau_plot) - 1, n_ticks).astype(int)
x_tick_positions = x_tick_indices + 0.5
x_tick_labels = [f"{tau_plot[i]:.2f}" for i in x_tick_indices]

plt.xticks(
    x_tick_positions,
    x_tick_labels,
    fontsize=16,
    rotation=0
)

# y-axis: lambda
lambda_plot = lambda_plot_raw[::-1]

y_tick_indices = np.linspace(0, len(lambda_plot) - 1, n_ticks).astype(int)
y_tick_positions = y_tick_indices + 0.5
y_tick_labels = [
    rf"$10^{{{np.log10(lambda_plot[i]):.1f}}}$"
    for i in y_tick_indices
]

plt.yticks(
    y_tick_positions,
    y_tick_labels,
    fontsize=16,
    rotation=0
)

# label for axis
tau_start = tau_plot[0]
tau_end = tau_plot[-1]
tau_num = len(tau_plot)

lambda_start = lambda_plot_raw[0]
lambda_end = lambda_plot_raw[-1]
lambda_num = len(lambda_plot_raw)

plt.title(
    r"Relative Error for NORA MC and target MC under scenario A ($\tau$ vs. $\lambda$)",
    fontsize=20,
    pad=24
)

plt.xlabel(
    rf"$\tau \in \mathrm{{linspace}}({tau_start:.2f}, {tau_end:.2f}, {tau_num})$",
    fontsize=18
)

plt.ylabel(
    rf"$\lambda \in \mathrm{{logspace}}({np.log10(lambda_start):.1f}, {np.log10(lambda_end):.1f}, {lambda_num})$",
    fontsize=18
)

for i in range(len(tau_plot) + 1):
    heatmap.axvline(i, color="white", linewidth=1)

for i in range(len(lambda_plot_raw) + 1):
    heatmap.axhline(i, color="white", linewidth=1)

plt.tight_layout()

# save image
plt.savefig(
    "heatmap.png",
    dpi=400,
    bbox_inches="tight"
)

plt.close()